# Week 4: Improved Model (v2)
## University Knowledge Assistant (RAG) · Group 4 · Baseline vs v1 vs v2

Week 3 left three open problems (see the v1 summary):

| Week-3 finding | v2 change |
|---|---|
| Right passage ranked first for only 74% of test questions (but in the top 5 for 98%) | **Retrieve-then-rerank**: hybrid retrieval gives 10 candidates, a **cross-encoder** re-scores (question, sentence) pairs |
| Small embedding model (MiniLM) | **Embedding model comparison on dev**: MiniLM-L6, bge-small, bge-base |
| Wrong sentence picked inside the right passage | Answer sentence chosen by the cross-encoder, optionally pooled over the top-3 chunks |
| Threshold tuned on 4 out-of-scope dev questions; 2 of 6 out-of-scope test questions answered | **Extended dev set** (new out-of-scope questions, auto-checked to be absent from the SRB) + **calibrated confidence** (logistic regression over lexical, semantic and cross-encoder signals, threshold on out-of-fold predictions) |
| Answers cut off at abbreviations such as \"(i.e.\" (found in live testing) | v2 sentence splitter protects abbreviations (v1 keeps the Week-3 splitter so it reproduces exactly) |


## 0. Setup

In [41]:
import importlib.util, subprocess, sys

packages = [("pymupdf", "pymupdf"), ("rank_bm25", "rank_bm25"), ("nltk", "nltk"), ("scikit-learn", "sklearn"),
            ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("scipy", "scipy"),
            ("sentence-transformers", "sentence_transformers"), ("python-docx", "docx")]
for pip_name, module in packages:
    if importlib.util.find_spec(module) is None:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
print("Python:", sys.executable)
print("All required packages are installed.")

Python: C:\Users\saniy\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe
All required packages are installed.


In [42]:
import os, re, time, json, unicodedata
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import pymupdf
except ImportError:
    import fitz as pymupdf

from rank_bm25 import BM25Okapi
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from scipy.stats import binomtest

import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
print("All imports successful!")

All imports successful!


In [ ]:
# Path to the SRB PDF (edit, or set the SRB_PDF_PATH environment variable)
PDF_PATH = os.environ.get("SRB_PDF_PATH",
    r"C:\Users\Sharvari\OneDrive\Documents\SNPL\Student Resource Book (SRB) 2026.pdf")

OUT_DIR, WEEK2_DIR, WEEK3_DIR = "week4_outputs", "week2_outputs", "week3_outputs"
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"          # v1 (Week 3)
CE_MODEL = os.environ.get("SRB_CE_MODEL", "cross-encoder/ms-marco-MiniLM-L-6-v2")
BGE_PREFIX = "Represent this sentence for searching relevant passages: "
SEED = 42

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED); torch.manual_seed(SEED)
pd.set_option("display.max_colwidth", 110)

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"Could not find the SRB PDF at: {PDF_PATH}. Edit PDF_PATH above.")
print("Using:", PDF_PATH)

Using: C:\Users\saniy\OneDrive\Documents\SNPL\Student Resource Book (SRB) 2026.pdf


## 1. Data (Week-2/3 pipeline, unchanged)

Cells 1.1-1.4 are copied from the Week-3 notebook without changes: load, clean, keep the 60 rule/policy pages, section tags, chunking, tokenisation, the 100 annotated QA pairs, and the same stratified 40/60 dev/test split (same seed).

### 1.1 Collection and cleaning

In [44]:
doc = pymupdf.open(PDF_PATH)
pages_raw = pd.DataFrame({
    "page": range(1, doc.page_count + 1),          # 1-based PDF page number (used for citations)
    "raw_text": [p.get_text() for p in doc],
})
pages_raw["n_words"] = pages_raw.raw_text.str.split().str.len()

print(f"Pages: {len(pages_raw)} | words: {pages_raw.n_words.sum():,}")
print("Pages with (almost) no text layer:", pages_raw[pages_raw.n_words < 20].page.tolist())
print("\nRaw sample (page 11):\n", pages_raw.raw_text[10][:400])

Pages: 172 | words: 60,404
Pages with (almost) no text layer: [2, 48, 66, 67, 169, 170]

Raw sample (page 11):
  
7 
3.2 
Attendance rules for all schools  
 
To ensure satisfactory attendance and engagement, 100% attendance in all subjects/courses is expected. However, exceptions 
may be made for valid medical reasons, personal reasons, and participation in extra-curricular and co-curricular activities, 
placement activities, institutional work, and other approved activities. A maximum relaxation of up to 


In [45]:
CHAR_MAP = {"’": "'", "‘": "'", "“": '"', "”": '"', "–": "-", "—": "-", "\u00a0": " "}

def clean_page(text):
    text = unicodedata.normalize("NFKC", text)
    for a, b in CHAR_MAP.items():
        text = text.replace(a, b)
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if lines and lines[0].isdigit():            # printed page number (top)
        lines = lines[1:]
    if lines and lines[-1].isdigit():           # printed page number (bottom)
        lines = lines[:-1]
    text = " ".join(lines)                              # re-join broken lines
    text = re.sub(r"_{3,}|[.…]{4,}", " ", text)         # form blanks
    text = re.sub(r"(\w)-\s+(\w)", r"\1-\2", text)      # split hyphenation
    text = re.sub(r"\s+([,;:])", r"\1", text)           # space before punctuation
    return re.sub(r"\s+", " ", text).strip()

pages_raw["clean_text"] = pages_raw.raw_text.map(clean_page)
print("BEFORE:", repr(pages_raw.raw_text[9][:250]))
print("\nAFTER: ", pages_raw.clean_text[9][:250])

BEFORE: ' \n6 \n3.0 \nAttendance guidelines and Attendance rules : 100% Attendance in each course is expected. \n \n3.1 Attendance guidelines: \n3.1.1 \nA student is required to monitor his /her own attendance regularly.  All doubts regarding attendance should be \nc'

AFTER:  3.0 Attendance guidelines and Attendance rules: 100% Attendance in each course is expected. 3.1 Attendance guidelines: 3.1.1 A student is required to monitor his /her own attendance regularly. All doubts regarding attendance should be clarified with 


In [46]:
EXCLUDE = {
    **{p: "Cover / part title page" for p in [1, 2, 67]},
    **{p: "Table of contents" for p in [4, 69]},
    **{p: "Annexures: blank application forms" for p in range(48, 67)},
    **{p: "Curriculum / course-list tables (later)" for p in range(83, 169)},
    **{p: "Academic calendar (image only, no text)" for p in [169, 170]},
}
pages_raw["status"] = pages_raw.page.map(EXCLUDE).fillna("KEPT: rules, policies, contacts")
pages = pages_raw[~pages_raw.page.isin(EXCLUDE)].copy()
assert (pages.clean_text.str.split().str.len() >= 20).all(), "a kept page is (almost) empty"

display(pages_raw.groupby("status").agg(pages=("page", "count"), words=("n_words", "sum")))
print(f"Knowledge base after cleaning: {len(pages)} pages, {pages.clean_text.str.split().str.len().sum():,} words")

,pages,words
status,,
"Academic calendar (image only, no text)",2,9
Annexures: blank application forms,19,3291
Cover / part title page,3,77
Curriculum / course-list tables (later),86,25451
"KEPT: rules, policies, contacts",60,31017
Table of contents,2,559


Knowledge base after cleaning: 60 pages, 30,916 words


### 1.2 Sections, chunking and tokenisation

In [47]:
# (PDF page, heading text as it appears on that page, section name)
SECTIONS = [
    (3, "Message from Vice-Chancellor", "Message from the Vice-Chancellor"),
    (5, "Student Guidelines", "1. About these Guidelines"),
    (5, "2 General Guidelines", "2. General Guidelines & Code of Conduct"),
    (10, "3.0 Attendance guidelines", "3. Attendance Rules"),
    (12, "4. Academic Guidelines", "4. Academic Guidelines"),
    (13, "5. Internal Continuous Assessment", "5. Internal Continuous Assessment (ICA)"),
    (14, "6. Interdisciplinary Offerings", "6. Interdisciplinary Offerings"),
    (14, "7. CHOICE BASED CREDIT SYSTEM", "7. Choice Based Credit System (CBCS)"),
    (15, "8. NMIMS-COURSERA INITIATIVE", "8. NMIMS-Coursera Initiative"),
    (16, "9. Examination Guidelines", "9. Examination Guidelines"),
    (23, "10. Policy on Award", "10. Awards, Prizes & Dean's List"),
    (24, "11. Guidelines for Scholarships", "11. Scholarships"),
    (25, "12. Guidelines for Admission Deferment", "12. Admission, Fees, Academic Break & Re-admission"),
    (28, "13. Guidelines for De-registration", "13. De-registration for Non-attendance"),
    (30, "14. Library Rules and Regulations", "14. Library Rules"),
    (31, "15. List of E-resources", "15. E-resources"),
    (32, "16. Guidelines for the Use of Computing Facilities", "16. Computing Facilities"),
    (34, "17. Students Portal", "17. Student Portal (LMS)"),
    (35, "18. Feedback Mechanism", "18. Feedback Mechanism"),
    (35, "19. Mentoring Programme", "19. Mentoring & Counselling"),
    (37, "20. Rules for Participating", "20. National/International Contests"),
    (37, "21 Guidelines for Annual Convocation", "21. Convocation"),
    (38, "22 Roles and Responsibilities", "22. Students' Council (NUSC)"),
    (39, "23 Fee-Related Procedures", "23. Fee-related Procedures & Refunds"),
    (40, "24 University level", "24. Anti-Ragging, ICC & Ombudsman"),
    (43, "25 Safety Guide", "25. Safety Guide (Floods, Fire, Earthquakes)"),
    (45, "26 The list of website categories", "26. Blocked Website Categories"),
    (46, "27 SVKM's NMIMS INFOLINE", "27. Infoline (Mumbai Campus)"),
    (47, "28 People you should know", "28. People You Should Know (University)"),
    (68, "Message from Dean", "MPSTME: Message from the Dean"),
    (70, "1. Academic/general inputs", "MPSTME 1. School Rules, Committees & CR"),
    (73, "2. Placement Guidelines", "MPSTME 2. Placements & Internships"),
    (76, "3. MPSTME International linkages", "MPSTME 3. International Linkages"),
    (77, "4. Examination Guidelines", "MPSTME 4. Passing Standards & Progression (ATKT)"),
    (79, "5. Promotion Rules: M. Tech. / MCA", "MPSTME 5. Promotion Rules (M Tech / MCA)"),
    (80, "6. Re-Examination Application", "MPSTME 6. Re-examination Fees"),
    (80, "7. Grading System", "MPSTME 7-8. Grading System & CGPA"),
    (81, "9. General Examination Rules", "MPSTME 9. General Examination Rules"),
    (82, "10. Admission Policy", "MPSTME 10-13. D.Voc Admission, Course Policy, Community Service"),
    (171, "17. LIST OF HOLIDAYS", "MPSTME 17. List of Holidays"),
    (172, "18. People you should know", "MPSTME 18. People You Should Know (School)"),
]

def split_into_sections(pages):
    """Cut each page at its known headings -> segments that never cross a page or section."""
    headings = defaultdict(list)
    for page, heading, name in SECTIONS:
        headings[page].append((heading, name))
    segments, current = [], "Front matter"
    for page, text in zip(pages.page, pages.clean_text):
        cuts = []
        for heading, name in headings[page]:
            pos = text.find(heading)
            assert pos >= 0, f"Heading '{heading}' not found on page {page}"
            cuts.append((pos, name))
        start = 0
        for pos, name in sorted(cuts):
            if text[start:pos].strip():
                segments.append((page, current, text[start:pos].strip()))
            current, start = name, pos
        segments.append((page, current, text[start:].strip()))
    return pd.DataFrame(segments, columns=["page", "section", "text"])

segments = split_into_sections(pages)
segments["part"] = np.where(segments.page < 67, "Part I (University)", "Part II (MPSTME)")
print(f"{len(segments)} segments in {segments.section.nunique()} sections")

70 segments in 41 sections


In [48]:
def make_chunks(segments, size=100, overlap=25):
    """Overlapping word windows inside each segment; every chunk keeps its page + section."""
    rows = []
    for page, section, part, text in segments[["page", "section", "part", "text"]].itertuples(index=False):
        words, start = text.split(), 0
        while True:
            rows.append({"page": page, "section": section, "part": part,
                         "text": " ".join(words[start:start + size])})
            if start + size >= len(words):
                break
            start += size - overlap
    chunks = pd.DataFrame(rows)
    chunks.insert(0, "chunk_id", [f"C{i:04d}" for i in range(len(chunks))])
    chunks["n_words"] = chunks.text.str.split().str.len()
    return chunks

STEM = PorterStemmer().stem
def tokenize(text):
    """lowercase -> alphanumeric tokens -> drop stop-words -> Porter stem"""
    return [STEM(t) for t in re.findall(r"[a-z0-9]+", text.lower()) if t not in ENGLISH_STOP_WORDS]

### 1.3 QA annotations (100 pairs, auto-validated)

Loaded from `week2_outputs/qa_annotations.csv` (the Week-2 benchmark). The embedded copy in the next cell is only a fallback if that file is missing.

In [49]:
QA_CSV = f"{WEEK2_DIR}/qa_annotations.csv"
QA_COLS = ["qid", "topic", "question", "answer", "answer_key", "evidence", "page"]
QA = None
if os.path.exists(QA_CSV):
    _q = pd.read_csv(QA_CSV, keep_default_na=False)          # keep empty strings of the out-of-scope rows
    if set(QA_COLS) <= set(_q.columns) and len(_q) == 100:
        QA = _q[QA_COLS].copy()
        print(f"Loaded {len(QA)} annotated questions from {QA_CSV}")
    else:
        print(f"{QA_CSV} has unexpected columns/size ({list(_q.columns)}, {len(_q)} rows): using the embedded fallback")
else:
    print(f"{QA_CSV} not found: using the embedded fallback")

Loaded 100 annotated questions from week2_outputs/qa_annotations.csv


Fallback copy of the annotations (only used if the CSV above could not be loaded).

In [50]:
# (topic, question, reference answer, answer key, evidence span, PDF page)
QA_FALLBACK = [
# ---------------- Attendance ----------------
("attendance", "What is the minimum attendance required to appear for the end-semester exam?",
 "You need at least 80% attendance in each individual course.", "80%",
 "Students who have maintained 80% or higher attendance in individual courses", 11),
("attendance", "I was marked absent even though I was in class. What should I do?",
 "Inform the concerned faculty immediately with a written application stating you were present, and keep a copy.", "written application",
 "he/she should immediately inform the concerned faculty by submitting a written application", 10),
("attendance", "How many days do I have to report a mistake in the monthly attendance report?",
 "Within 3 days of the attendance record being published; no claims are accepted after that.", "3 days",
 "within 3 days of the publication of the attendance record", 10),
("attendance", "When will my parents be informed about my attendance?",
 "If your monthly attendance is below 80%, parents are informed by email/WhatsApp.", "less than 80%",
 "if the student has less than 80% attendance monthly through email/WhatsApp", 10),
("attendance", "By when should I submit a medical certificate after coming back from medical leave?",
 "Within 3 days of resuming classes, along with copies of all medical reports.", "within 3 days of resuming",
 "within 3 days of resuming classes after medical leave", 10),
("attendance", "What happens if my attendance in a subject goes below 80%?",
 "You will have to take re-admission, either in the same semester or for the full year in the next academic year.", "re-admission",
 "Students will be required to take re-admission. Students can opt", 11),
("attendance", "Am I marked present in regular classes on the day I write a re-exam?",
 "Yes. You are marked present for all applicable subjects on the day of your re-examination.", "marked present",
 "such students shall be marked present for all applicable subjects on that day", 10),
("attendance", "How much absence can be relaxed for medical reasons or extra-curricular activities?",
 "Up to 20% absence can be relaxed, as long as attendance stays at 80% or above in each course.", "20%",
 "A maximum relaxation of up to 20% absence may be granted", 11),
# ---------------- Academics & ICA ----------------
("academics_ica", "What is the minimum duration of a term end exam?",
 "At least 2 hours.", "2 hrs",
 "Minimum duration of Term End Examinations: 2 hrs", 13),
("academics_ica", "What do I need to score to pass a course that has only ICA and no final exam?",
 "At least 40% of the total ICA marks.", "40%",
 "The passing criteria for such ONLY ICA courses will be at 40% minimum of total of ICA marks", 13),
("academics_ica", "Is there a re-exam if I fail an only-ICA course?",
 "No. There is no re-exam for only-ICA courses; you must take re-admission.", "no re-examination",
 "there will be no re-examination for Only ICA Courses", 13),
("academics_ica", "Can grace marks be added to my internal (ICA) marks?",
 "No. Grace marks are never awarded on the ICA component.", "never",
 "Grace Marks will never be awarded on the ICA (Internal Continuous Assessment) component|Grace marks will never to be awarded on ICA", 13),
("academics_ica", "How long do I have to raise an issue about my ICA marks on the portal?",
 "Within two days of the marks being published.", "two days",
 "should be informed and resolved within two days of publication on the portal", 13),
("academics_ica", "If I am caught copying in an assignment and allowed to resubmit, how many marks can I get?",
 "50% of the total marks or your actual marks, whichever is lower (for group work this applies to everyone).", "50%",
 "the student will be awarded 50% of the total marks or actual marks, whichever is lower", 14),
("academics_ica", "How many interdisciplinary courses can I take?",
 "A maximum of 2: one in the fall session and one in the winter session.", "maximum of 2",
 "allowed to choose a maximum of 2 courses", 14),
("academics_ica", "Can first-year undergraduate students take CBCS courses?",
 "No. First-year UG students are not eligible for CBCS courses.", "not eligible",
 "are not eligible to opt CBCS courses", 14),
("academics_ica", "How long is the Coursera license valid?",
 "For one academic year only.", "one academic year",
 "Coursera license which will be valid for one academic year only", 15),
("academics_ica", "Where do I email an academic grievance?",
 "Write to grievance.academic@nmims.edu.", "grievance.academic@nmims.edu",
 "the student may write to grievance.academic@nmims.edu", 15),
# ---------------- Exam conduct & unfair means ----------------
("exam_conduct", "How early should I reach the exam hall?",
 "At least 30 minutes before the exam starts.", "30 minutes before",
 "at least 30 minutes before the scheduled time of the commencement of the examination", 16),
("exam_conduct", "Can I still write the exam if I come more than 30 minutes late?",
 "No. Students arriving after 30 minutes are not allowed under any circumstances.", "not be allowed",
 "will not be allowed to write examination under any circumstances", 16),
("exam_conduct", "Is a smart watch allowed in the exam hall?",
 "No. Smart watches and similar devices are forbidden and will be confiscated.", "smart watch",
 "notes, scribbling papers, mobile phones, smart watch", 16),
("exam_conduct", "Can I write my name or roll number on the answer book?",
 "No. Answer books are QR coded; revealing your identity is treated as unfair means.", "qr coded|not write",
 "The answer books of the term-end examinations will be QR coded", 17),
("exam_conduct", "What is the punishment for impersonation in an exam?",
 "Permanent expulsion from the University.", "permanent expulsion",
 "Impersonation for a student or impersonation by a student in University or other examinations", 19),
("exam_conduct", "What happens if a student is caught using unfair means more than twice?",
 "Permanent expulsion from the University.", "permanent expulsion",
 "Student involved in uses of Unfair Means more than two occasions", 20),
("exam_conduct", "What is the punishment for trying to bribe an examiner?",
 "Exclusion from four additional examinations; the money goes to the Students Welfare Fund.", "four additional examinations",
 "Insertion of currency notes/ bribing or attempt to bribe", 19),
# ---------------- Exam support (scribe / LD) ----------------
("exam_support", "How much extra time does a student with a scribe get?",
 "20 minutes per hour (e.g. 40 minutes for a 2-hour exam).", "20 minutes per hour",
 "compensatory time of 20 minutes per hour will be allowed", 17),
("exam_support", "How early do I need to arrange a scribe for my exam?",
 "At least one week before the exam, and inform the examination office.", "one week",
 "at least one week before the examination", 17),
("exam_support", "Which hospitals can issue the learning disability certificate for the Mumbai campus?",
 "Only Sion Hospital or Nair Hospital.", "sion hospital|nair hospital",
 "medical certificate from Sion Hospital / Nair Hospital only would be accepted", 18),
("exam_support", "Can students with dyslexia get grace marks?",
 "Yes, up to 3% of the aggregate marks of the subjects appeared in.", "3 per cent",
 "eligible for grace marks up to 3 per cent of the aggregate", 18),
# ---------------- Results, revaluation & grace ----------------
("results_revaluation", "Within how many days of the result can I ask for a photocopy of my answer sheet?",
 "Within 2 days from the date of result declaration (including holidays).", "2 days",
 "Application for Photocopy/ Softcopy of the Answer Script Within 2 days", 21),
("results_revaluation", "What is the deadline to apply for retotalling of marks?",
 "Within 3 days from the date of result declaration (including holidays).", "3 days",
 "Application for Verification (retotalling of marks)", 21),
("results_revaluation", "Can I apply for revaluation without taking a photocopy of my answer script?",
 "No. You must first apply for the photocopy to be eligible for revaluation.", "not be eligible",
 "Students will not be eligible to apply for revaluation if they do not apply for a Photocopy", 21),
("results_revaluation", "What happens if revaluation changes my marks by more than 20%?",
 "A second revaluation is done and the better of the two revaluations becomes final.", "second revaluation",
 "Second revaluation of the said answer book shall be done", 22),
("results_revaluation", "How long does the whole grievance redressal process take?",
 "It is completed within 15 working days of receiving the application.", "15 working days",
 "completed within a period of 15 working days", 23),
("results_revaluation", "How many grace marks can a student who fails get?",
 "Up to 2% of aggregate marks, maximum 6 marks overall, and not more than 3% per subject.", "2 per cent|6 marks",
 "grace marks up to 2 per cent of the marks on the aggregate marks", 23),
("results_revaluation", "Can I apply for revaluation of my viva or lab exam?",
 "No. Grievance redressal applies only to term-end exams, not practicals, labs, viva or projects.", "will not apply",
 "The above mechanism will not apply to practicals / Labs / oral examinations", 21),
# ---------------- Awards & scholarships ----------------
("awards_scholarships", "What percentage of students make it to the Dean's list?",
 "The top 10% of the batch by CGPA.", "10%",
 "10% of the batch (top 10% students) on the basis of highest CGPA", 23),
("awards_scholarships", "How many prizes are given if my batch has 30 students?",
 "Two prizes (for batches of 21 to 59 students).", "two prizes",
 "two prizes will be awarded where the number of students in the batch/stream is between 21 and 59", 23),
("awards_scholarships", "Can a student with an ATKT be on the rank holder list?",
 "No. Students with F grades/ATKT or misconduct are ineligible.", "ineligible",
 "Students obtaining F grades/ATKT/ involved in unfair means or any misconduct will be ineligible", 23),
("awards_scholarships", "Who gets disqualified from the awards and scholarships process?",
 "Students with an F grade, a misconduct record or low attendance are automatically disqualified.", "automatically disqualified",
 "will be automatically disqualified from the awards process", 24),
("awards_scholarships", "Where can I apply for government scholarships?",
 "Through the National Scholarship Portal (NSP), scholarships.gov.in.", "national scholarship portal|nsp",
 "can avail national scholarship through the National Scholarship Portal (NSP)", 24),
("awards_scholarships", "Is there any scholarship for girls in technical courses?",
 "Yes, the AICTE Pragati Scholarship Scheme for Girl Students, via NSP.", "pragati",
 "Pragati Scholarship Scheme for Girl Students", 24),
# ---------------- Admission, academic break & re-admission ----------------
("admission_readmission", "For what reasons can I defer my admission?",
 "Serious medical illness or serious family-related reasons.", "medical illness|family-related",
 "Serious medical illness", 25),
("admission_readmission", "Can a second-year student get admission deferment?",
 "No. Admission deferment can be approved only for the first year.", "only for first year|first-year students",
 "Admission deferment can be approved only for first year|Deferred admission may only be granted to first-year students", 25),
("admission_readmission", "On what grounds can I take an academic break?",
 "Serious medical reasons, serious family issues, or financial constraints.", "medical|financial constraints",
 "Serious personal medical reasons involving hospitalisation", 26),
("admission_readmission", "How many times can I take an academic break during my program?",
 "At most twice, as long as the total break does not exceed one year.", "twice",
 "granted to any student at best twice during the programme", 26),
("admission_readmission", "How much is the re-admission fee?",
 "25% of the total fee prevailing at the time of re-admission.", "25%",
 "he / she has to pay 25% of the total fee prevalent", 27),
("admission_readmission", "Can the 25% re-admission fee ever be waived?",
 "Yes, if the break is on health grounds and attendance was below 20%.", "waived",
 "the 25% fee for re-admission could be waived off", 27),
("admission_readmission", "How many times can I take re-admission in the same year of my program?",
 "Only once.", "only once",
 "Students can take re-admission in the said year of the program only once", 27),
("admission_readmission", "What happens if I don't pay my fees even after the late-fee period?",
 "Your studentship in the program is cancelled.", "cancellation of the studentship",
 "Non-payment of fees within the stipulated time, including the late fee period", 26),
# ---------------- Conduct & discipline ----------------
("conduct_discipline", "Are mobile phones allowed on campus?",
 "No. Mobile phones, smart glasses and similar devices are not permitted on campus.", "not permitted",
 "The use of mobile phones, Meta Smart Glasses, or any similar electronic devices on campus is not permitted", 5),
("conduct_discipline", "Can I eat in the classroom?",
 "No. Only drinking water is allowed inside the classroom.", "drinking water",
 "Only drinking water is allowed inside the classroom", 5),
("conduct_discipline", "What should I do if I forget my ID card?",
 "Approach security and complete the process to get access; ID grievances go to biometricregistration@svkm.ac.in.", "approach security",
 "In case a student forgets to carry an ID card, they should approach security", 6),
("conduct_discipline", "What is the dress code for university events and guest lectures?",
 "Formals with the institute blazer, tie/cravat and lapel pin.", "blazer",
 "dress in formals, Institute blazer, Tie/ Cravat, and Lapel Pin", 9),
("conduct_discipline", "Who hears appeals in disciplinary cases?",
 "The Chancellor of the University is the appellate authority.", "chancellor",
 "the Chancellor of the University will be the appellate authority", 9),
("conduct_discipline", "Who sits on the school's disciplinary committee?",
 "The Chairperson/Head of Department, one faculty member and one staff member.", "one staff member",
 "The committee comprises the Chairperson / Head of the Department, one faculty member and one staff member", 9),
("conduct_discipline", "Do I need permission to travel out of the city during the term?",
 "Yes, prior permission from the HOD/Director/Dean is required.", "prior permission",
 "they are required to obtain prior permission from the HOD/Director/Dean", 9),
("conduct_discipline", "Where do I fill the anti-ragging undertaking?",
 "Online at www.antiragging.in or www.amanmovement.org.", "antiragging.in|amanmovement.org",
 "www.amanmovement.org (OR) www.antiragging.in", 40),
("conduct_discipline", "Who is the chairperson of the MPSTME anti-ragging committee?",
 "Dr. Koteswararao Anne.", "koteswararao anne",
 "1.5.1 Anti-Ragging Committee 1. Dr. Koteswararao Anne Chairperson", 71),
("conduct_discipline", "Who heads the University's Internal Complaints Committee for sexual harassment?",
 "Dr. Meena Galliara (Chairperson).", "meena galliara",
 "1. Dr. Meena Galliara, Director, (Jasani Centre", 41),
# ---------------- Library ----------------
("library", "How many library books can I borrow and for how long?",
 "Two books for ten days, with one renewal if the book is not in demand.", "two books for ten days",
 "Borrowing entitlement: Two books for ten days", 30),
("library", "What is the fine for returning a library book late?",
 "Rs. 3 per day per book.", "rs.3.00|rs. 3",
 "A fine of Rs.3.00 per day per book is levied on overdue books", 30),
("library", "What happens if I use my phone in the library?",
 "It can lead to a fine, exclusion from the library and/or a three-week suspension.", "three week suspension",
 "Violating the rules will lead to a student's penalty and /or three week suspension", 30),
("library", "Will I get my degree if I still have library books or fines pending?",
 "No. The award of the qualification is deferred until all books are returned and fines paid.", "deferred",
 "The award of SVKM's NMIMS qualification will be deferred until all books", 30),
# ---------------- IT & student portal ----------------
("it_portal", "How much OneDrive storage do students get?",
 "50 GB.", "50 gb",
 "Students will get 50 GB of space in OneDrive", 32),
("it_portal", "How early should I ask IT for extra internet bandwidth for an event?",
 "At least 72 hours in advance.", "72 hours",
 "a request should reach the IT helpdesk minimum 72 hours in advance", 33),
("it_portal", "What is my default username for the student portal?",
 "Your SAP number; the password is mailed by the admin/course coordinator.", "sap number",
 "Default User ID is the student's SAP number", 34),
("it_portal", "Can I plug my pen drive into the classroom smartboard?",
 "No. Personal pen drives and laptops cannot be connected to classroom systems/smartboards.", "not allowed",
 "Students are not allowed to connect personal pen drives/ Laptops to the systems/smartboard", 32),
("it_portal", "How do I get access to a website that is blocked on the campus network?",
 "Email ITHelpdesk@nmims.edu; access is granted if the content is found appropriate.", "ithelpdesk@nmims.edu",
 "request to send an email to ITHelpdesk@nmims.edu", 32),
("it_portal", "Can I book a hostel room through the student portal?",
 "Yes, hostel booking is available on the portal once online admission and full payment are done.", "book a hostel",
 "Students can book a hostel through the portal", 34),
# ---------------- Student services, fees & refunds ----------------
("student_services", "Do I get travel reimbursement for a Grade B contest?",
 "Only if you win 1st or 2nd place; then travel is reimbursed 100% (non-AC sleeper/3-tier).", "1st or 2nd",
 "provided that they have won the contest (1st or 2nd place only)", 37),
("student_services", "How soon after winning a contest must I submit the details?",
 "Within 7 days of winning, or it won't be considered for appraisal.", "7 days",
 "within 7 days of winning the contest", 37),
("student_services", "How long does it take to get my hostel deposit refunded?",
 "Allow 3 weeks after submitting the signed receipt and refund application to Accounts.", "3 weeks",
 "procure the signature of the Hostel in-charge on the receipt", 39),
("student_services", "How quickly must expenses for a student activity be settled?",
 "Within 72 hours, along with the activity report.", "72 hours",
 "they must be settled within 72 hours along with the report of activities", 39),
("student_services", "How can MPSTME students reach the university psychologist?",
 "Call 022-42332225 / 022-42334090 or email Nazneen.raimalwala@nmims.edu or ketaki.gokhale@nmims.edu.", "nazneen.raimalwala@nmims.edu|ketaki.gokhale@nmims.edu",
 "email Nazneen.raimalwala@nmims.edu", 36),
("student_services", "If I'm found ineligible for my degree at convocation, by when can I appeal?",
 "At least 48 hours before the convocation ceremony.", "48 hours",
 "at least 48 hours before the Annual Convocation", 37),
("student_services", "Is Maharashtra Day a university holiday?",
 "Yes, 1 May (Maharashtra Day) is a holiday and one of the national holidays when no classes are held.", "maharashtra day",
 "01-May-25 Thursday Maharashtra Day", 171),
# ---------------- MPSTME: passing, progression, re-exams, grading ----------------
("mpstme_exams", "What do I need to score to pass a course?",
 "At least 40% in the semester-end exam and at least 40% aggregate (ICA + exam).", "40%",
 "Secure a minimum of 40% marks in the Term / Semester end examination of that course", 77),
("mpstme_exams", "How many F grades can I have and still move to the next year in B Tech?",
 "Up to two F grades after re-exams; you then re-appear for those courses next year.", "up to two",
 "A student who has obtained up to two 'F' grades in B Tech / MBA Tech/ Diploma program", 78),
("mpstme_exams", "What happens if I get more than two F grades in a year?",
 "You are not promoted; you can take re-admission for the whole year or take a 'year back' and re-appear only for the failed exams.", "not be promoted",
 "would not be promoted to the next year of the programme", 78),
("mpstme_exams", "What is the re-exam fee per subject?",
 "Rs. 1,000 per subject.", "1,000",
 "The fee for submission of a re-examination application shall be Rs. 1,000/-", 80),
("mpstme_exams", "What is the late fee if I miss the re-exam application deadline?",
 "Rs. 5,000 (in addition to Rs. 1,000 per subject), payable in a 24-hour late window.", "5,000|5000",
 "allowed to submit their application with late fees of Rs. 5000/-", 80),
("mpstme_exams", "For how many days is the re-exam application portal open?",
 "Three days; it closes at 4:00 PM on the third day.", "three (3) days|4:00 pm",
 "will remain open for three (3) days", 80),
("mpstme_exams", "How are the marks split between internals and the semester-end exam?",
 "ICA is 50 marks and the semester-end exam is 100 marks scaled down to 50, for a total of 100.", "scaled down to 50",
 "100 (scaled down to 50 marks)", 77),
("mpstme_exams", "What happens if I am absent for a semester-end exam?",
 "You are marked 'AB' for that course and can appear for the re-examination.", "marked as 'ab'",
 "shall be marked as 'AB' in the result", 81),
# ---------------- MPSTME: placements, internships & school ----------------
("mpstme_school", "How long is the internship for B Tech students?",
 "An entire semester (the last semester of the year).", "entire semester",
 "Internship duration for B Tech, and MCA - entire semester", 74),
("mpstme_school", "What if I don't want to sit for campus placements?",
 "Formally notify the Placement Office using the 'Opted Out Form' with your reason.", "opted out form",
 "formally notify the Placement Office vide the 'Opted Out Form'", 74),
("mpstme_school", "What CGPA does a D.Voc student need to join B Tech first year?",
 "A minimum of 6.5 based on first- and second-year exams.", "6.5",
 "Candidates must have secured a minimum CGPS of 6.5", 82),
("mpstme_school", "When is the community service course done?",
 "Any three consecutive weeks in the summer vacation between Semester II and Semester III.", "three weeks",
 "any consecutive three weeks during the summer vacation between Semester-II and Semester-III", 82),
("mpstme_school", "How is the class representative (CR) chosen?",
 "The top three students by Class XII aggregate are nominated; the HoD selects the final CR after interaction.", "class xii",
 "The first three students, based on their Class XII aggregate marks", 72),
# ---------------- Out of scope (answer is NOT in the SRB) ----------------
("out_of_scope", "What is the hostel curfew time for students?", "", "", "", None),
("out_of_scope", "What is the annual tuition fee for B Tech Data Science?", "", "", "", None),
("out_of_scope", "What is the campus Wi-Fi password?", "", "", "", None),
("out_of_scope", "What time does the college canteen close?", "", "", "", None),
("out_of_scope", "Is there a gym on campus and what are its timings?", "", "", "", None),
("out_of_scope", "What is the average placement package for B Tech students?", "", "", "", None),
("out_of_scope", "When is the MPSTME college fest held?", "", "", "", None),
("out_of_scope", "What is the NPAT cutoff score for B Tech admission?", "", "", "", None),
("out_of_scope", "Is there a college shuttle bus from the railway station?", "", "", "", None),
("out_of_scope", "How much is the parking fee for two-wheelers on campus?", "", "", "", None),
]

In [51]:
if QA is None:
    qa = pd.DataFrame(QA_FALLBACK, columns=["topic", "question", "answer", "answer_key", "evidence", "page"])
    qa.insert(0, "qid", [f"Q{i:03d}" for i in range(1, len(qa) + 1)])
else:
    qa = QA.copy()
qa["answerable"] = qa.evidence != ""
qa["page"] = pd.to_numeric(qa.page, errors="coerce").astype("Int64")

def norm(s):
    """Same cleaning as the corpus + lowercase, so evidence/keys can be matched as substrings."""
    return clean_page(s).lower()

# ---- automatic annotation checks ----
page_text = dict(zip(pages.page, pages.clean_text.map(norm)))
problems = []
for r in qa.itertuples():
    if not r.answerable:
        if r.answer_key or pd.notna(r.page):
            problems.append((r.qid, "out-of-scope item should have no key/page"))
        continue
    spans = [norm(e) for e in r.evidence.split("|")]
    evidence_pages = [p for p, t in page_text.items() if any(e in t for e in spans)]
    if spans[0] not in page_text.get(r.page, ""):
        problems.append((r.qid, "evidence not found on the annotated page"))
    if any(not any(e in t for t in page_text.values()) for e in spans):
        problems.append((r.qid, "an evidence span is not in the corpus"))
    if not any(k.strip().lower() in page_text[p] for p in evidence_pages for k in r.answer_key.split("|")):
        problems.append((r.qid, "answer key not found next to the evidence"))
    if any(len(e.split()) > 20 for e in spans):
        problems.append((r.qid, "evidence longer than 20 words"))
if qa.question.duplicated().any():
    problems.append(("-", "duplicate questions"))
assert not problems, problems
print(f"All {len(qa)} annotations passed: {qa.answerable.sum()} answerable + {(~qa.answerable).sum()} out-of-scope")

All 100 annotations passed: 90 answerable + 10 out-of-scope


### 1.4 Dev / test split (identical to Weeks 2-3)

In [52]:
dev, test = train_test_split(qa, test_size=0.60, stratify=qa.topic, random_state=SEED)
dev = dev.sort_values("qid").reset_index(drop=True)
test = test.sort_values("qid").reset_index(drop=True)
assert set(dev.qid).isdisjoint(test.qid) and len(dev) + len(test) == len(qa)

# If the Week-2 outputs are next to this notebook, prove the split is exactly the same
if os.path.exists(f"{WEEK2_DIR}/qa_test.csv"):
    assert pd.read_csv(f"{WEEK2_DIR}/qa_test.csv").qid.tolist() == test.qid.tolist()
    assert pd.read_csv(f"{WEEK2_DIR}/qa_dev.csv").qid.tolist() == dev.qid.tolist()
    print("Split matches week2_outputs/qa_dev.csv and qa_test.csv")
print(f"dev : {len(dev)} questions ({dev.answerable.sum()} answerable, {(~dev.answerable).sum()} out-of-scope)")
print(f"test: {len(test)} questions ({test.answerable.sum()} answerable, {(~test.answerable).sum()} out-of-scope)")

Split matches week2_outputs/qa_dev.csv and qa_test.csv
dev : 40 questions (36 answerable, 4 out-of-scope)
test: 60 questions (54 answerable, 6 out-of-scope)


### 1.5 Extended dev set (new in Week 4)

Week 3 tuned the refusal threshold on only **4** out-of-scope questions. We add new out-of-scope questions to the **dev set only**. Each one is paired with a term that must not occur anywhere in the knowledge base; the cell checks this automatically and drops any question whose term does occur (so no "out-of-scope" question can secretly be answerable). The **test set is not changed**.

In [53]:
EXTRA_OOS = [  # (question, regex that must NOT occur in the knowledge base)
    ("How can I book a badminton court on campus?", r"badminton"),
    ("Does the university provide a laundry service?", r"laundry"),
    ("What is on the cafeteria menu this week?", r"\bmenu\b"),
    ("Is there an ATM inside the campus?", r"\batm\b"),
    ("Which cab services operate near the campus?", r"\bcabs?\b"),
    ("What is the NAAC grade of the university?", r"naac"),
    ("What is the university's QS world ranking?", r"\bqs\b"),
    ("Is there a swimming pool on campus?", r"swimming"),
    ("What is the monthly bus fare to Andheri?", r"\bfare\b"),
    ("What are the opening hours of the stationery shop?", r"stationer"),
    ("Can I bring my pet to campus?", r"\bpets?\b"),
    ("Who is the football coach of the university team?", r"football"),
    ("How do I apply for a student visa for an exchange programme?", r"\bvisa\b"),
    ("What is the annual hostel mess charge?", r"\bmess\b"),
    ("Does the university offer a PhD in Chemistry?", r"chemistry"),
    ("What is the stipend for the summer internship?", r"stipend"),
]
kb_text = " ".join(pages.clean_text).lower()
kept = [q for q, rx in EXTRA_OOS if not re.search(rx, kb_text)]
dropped = [q for q, rx in EXTRA_OOS if re.search(rx, kb_text)]
assert len(kept) >= 8, "too few valid out-of-scope questions"
assert not set(kept) & set(qa.question), "duplicate of an existing question"
extra_oos = pd.DataFrame({"qid": [f"X{i:03d}" for i in range(1, len(kept) + 1)], "topic": "out_of_scope",
                          "question": kept, "answer": "", "answer_key": "", "evidence": "",
                          "page": pd.array([pd.NA] * len(kept), dtype="Int64"), "answerable": False})
dev_ext = pd.concat([dev, extra_oos[dev.columns]], ignore_index=True)
print(f"Added {len(kept)} out-of-scope dev questions (dropped as present in the SRB: {dropped or 'none'})")
print(f"dev_ext: {len(dev_ext)} questions ({dev_ext.answerable.sum()} answerable, {(~dev_ext.answerable).sum()} out-of-scope)")
assert set(dev_ext.qid).isdisjoint(test.qid)

Added 15 out-of-scope dev questions (dropped as present in the SRB: ['What are the opening hours of the stationery shop?'])
dev_ext: 55 questions (36 answerable, 19 out-of-scope)


## 2. Baseline and v1 reproduced (frozen settings)

The Week-2 baseline (chunk size 150, threshold 0.55) and Week-3 v1 (chunk size 100, beta 0.6, threshold 0.48) are rebuilt with their frozen settings. We check that they give exactly the published test numbers. `run_eval` logs which questions were used at which stage, so §9 can prove that no test question was used for tuning.

In [54]:
class BM25Retriever:
    def __init__(self, chunks):
        self.ids = chunks.chunk_id.tolist()
        self.bm25 = BM25Okapi([tokenize(t) for t in chunks.text])

    def search(self, query, k=5):
        scores = self.bm25.get_scores(tokenize(query))
        top = np.argsort(-scores, kind="stable")[:k]
        return [(self.ids[i], float(scores[i])) for i in top]


class TfidfRetriever:
    def __init__(self, chunks):
        self.ids = chunks.chunk_id.tolist()
        self.vectorizer = TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None)
        self.matrix = self.vectorizer.fit_transform(chunks.text)

    def search(self, query, k=5):
        scores = cosine_similarity(self.vectorizer.transform([query]), self.matrix).ravel()
        top = np.argsort(-scores, kind="stable")[:k]
        return [(self.ids[i], float(scores[i])) for i in top]

In [55]:
# sentence boundaries: end punctuation, bullets, or clause numbers like "3.1.4"
SENT_SPLIT = re.compile(r"(?<=[.;?!])\s+(?=[A-Z(•'\"])|\s+(?=•)|\s+(?=\d{1,2}(?:\.\d{1,2}){1,3}\s)")

def split_sentences(text):
    parts = [s.strip(" •") for s in SENT_SPLIT.split(text)]
    return [s for s in parts if len(s.split()) >= 3] or [text]

def extract_answer(question, passage):
    """Return the passage sentence that shares the most keywords with the question (verbatim)."""
    q_terms = set(tokenize(question))
    sents = split_sentences(passage)
    best = int(np.argmax([len(q_terms & set(tokenize(s))) for s in sents]))
    if len(sents[best].split()) < 8 and best + 1 < len(sents):      # too short -> include next sentence
        start = passage.find(sents[best])
        return passage[start: passage.find(sents[best + 1], start) + len(sents[best + 1])]
    return sents[best]


class KeywordQA:
    """Week-2 baseline: keyword retrieval + extractive answer + source + confidence."""
    NOT_FOUND = "I could not find this in the Student Resource Book."

    def __init__(self, chunks, retriever=BM25Retriever, threshold=0.0, k=5):
        self.chunks = chunks.set_index("chunk_id")
        self.retriever = retriever(chunks)
        self.threshold, self.k = threshold, k
        # IDF weights for the confidence score (rare words matter more)
        doc_freq = Counter(t for text in chunks.text for t in set(tokenize(text)))
        n = len(chunks)
        self.idf = {t: np.log((n + 1) / (df + 1)) + 1 for t, df in doc_freq.items()}
        self.unseen_idf = np.log(n + 1) + 1                   # word that never appears in the SRB

    def confidence(self, question, passage):
        q_terms = set(tokenize(question))
        if not q_terms:
            return 0.0
        weight = {t: self.idf.get(t, self.unseen_idf) for t in q_terms}
        found = q_terms & set(tokenize(passage))
        return sum(weight[t] for t in found) / sum(weight.values())

    def ask(self, question):
        hits = self.retriever.search(question, self.k)
        top = self.chunks.loc[hits[0][0]]
        confidence = self.confidence(question, top.text)
        extracted = extract_answer(question, top.text)
        abstained = confidence < self.threshold
        return {
            "answer": self.NOT_FOUND if abstained else extracted,
            "source": f"SRB 2026 | {top.part} | {top.section} | PDF page {top.page}",
            "confidence": round(confidence, 2),
            "supporting_text": top.text,
            # extra fields used only for evaluation
            "extracted": extracted, "abstained": abstained,
            "cited_page": int(top.page), "ranked_ids": [c for c, _ in hits],
        }

def show(out, question):
    print("Q:", question)
    for field in ["answer", "source", "confidence"]:
        print(f"  {field:<15}: {out[field]}")
    print(f"  supporting_text: {out['supporting_text'][:220]}...\n")

In [56]:
def key_match(text, key):
    return any(k.strip().lower() in norm(text) for k in key.split("|") if k.strip())

def evaluate(system, qa_df, chunks):
    norm_chunks = list(zip(chunks.chunk_id, chunks.text.map(norm)))
    chunk_page = dict(zip(chunks.chunk_id, chunks.page))
    rows = []
    for r in qa_df.itertuples():
        out = system.ask(r.question)
        row = {"qid": r.qid, "topic": r.topic, "answerable": r.answerable, "question": r.question,
               "gold_answer": r.answer, "pred_answer": out["answer"], "source": out["source"],
               "confidence": out["confidence"], "abstained": out["abstained"],
               "supporting_text": out["supporting_text"]}
        if r.answerable:
            spans = [norm(e) for e in r.evidence.split("|")]
            gold = {cid for cid, t in norm_chunks if any(e in t for e in spans)}
            assert gold, f"{r.qid}: evidence not inside any chunk"
            gold_pages = set(chunk_page[c] for c in gold)
            rank = next((i + 1 for i, c in enumerate(out["ranked_ids"]) if c in gold), np.nan)
            row.update({"rank": rank,
                        "citation_ok": out["cited_page"] in gold_pages,
                        "key_in_passage": key_match(out["supporting_text"], r.answer_key),
                        "answer_ok_raw": key_match(out["extracted"], r.answer_key),     # before abstention
                        "answer_ok": (not out["abstained"]) and key_match(out["answer"], r.answer_key)})
        rows.append(row)
    res = pd.DataFrame(rows)
    for col in ["citation_ok", "key_in_passage", "answer_ok_raw", "answer_ok"]:
        res[col] = res[col].eq(True)          # out-of-scope rows -> False (they are filtered out anyway)
    return res

def summarize(res):
    a, o = res[res.answerable], res[~res.answerable]
    rank = a["rank"].astype(float)
    return pd.Series({
        "Hit@1": (rank == 1).mean(),
        "Hit@3": (rank <= 3).mean(),
        "Hit@5": rank.notna().mean(),
        "MRR@5": (1 / rank).fillna(0).mean(),
        "Citation accuracy (page)": a.citation_ok.mean(),
        "Answer accuracy (key fact)": a.answer_ok.mean(),
        "Answer accuracy if never refusing": a.answer_ok_raw.mean(),
        "Key fact in supporting text": a.key_in_passage.mean(),
        "False abstention (answerable)": a.abstained.mean(),
        "Hallucination rate (out-of-scope)": 1 - o.abstained.mean(),
    })

In [57]:
EVAL_LOG = []                                      # (stage, qids) for every evaluation run
def run_eval(system, qa_df, chunks, stage):
    EVAL_LOG.append((stage, set(qa_df.qid)))
    return evaluate(system, qa_df, chunks)

BASE_SIZE, BASE_THRESHOLD = 150, 0.55              # chosen on dev in Week 2
base_chunks = make_chunks(segments, BASE_SIZE, overlap=max(20, BASE_SIZE // 4))
baseline = KeywordQA(base_chunks, BM25Retriever, threshold=BASE_THRESHOLD)
res_test_base = run_eval(baseline, test, base_chunks, "final-test")

WEEK2_TEST = {"Hit@1": 0.667, "Hit@3": 0.889, "Hit@5": 0.926, "MRR@5": 0.772,
              "Citation accuracy (page)": 0.815, "Answer accuracy (key fact)": 0.370,
              "Answer accuracy if never refusing": 0.500, "Hallucination rate (out-of-scope)": 0.167}
base_summary = summarize(res_test_base)
mismatch = {k: (round(base_summary[k], 3), v) for k, v in WEEK2_TEST.items() if round(base_summary[k], 3) != v}
assert not mismatch, mismatch
print("Baseline reproduces the Week-2 test results exactly.")
base_summary.round(3).to_frame("baseline (test)")

Baseline reproduces the Week-2 test results exactly.


,baseline (test)
Hit@1,0.667
Hit@3,0.889
Hit@5,0.926
MRR@5,0.772
Citation accuracy (page),0.815
Answer accuracy (key fact),0.370
Answer accuracy if never refusing,0.500
Key fact in supporting text,0.796
False abstention (answerable),0.222
Hallucination rate (out-of-scope),0.167


In [58]:
encoder = SentenceTransformer(EMB_MODEL, device="cpu")     # CPU: same numbers on every machine
encoder.eval()

def embed(texts):
    """texts -> L2-normalised embeddings (n, 384); dot product = cosine similarity"""
    return encoder.encode(list(texts), batch_size=64, normalize_embeddings=True,
                          convert_to_numpy=True, show_progress_bar=False)

EMB_DIM = embed(["test"]).shape[1]
print(f"Loaded {EMB_MODEL}: {EMB_DIM}-dim embeddings, max {encoder.max_seq_length} tokens")

# Semantic similarity in action: same meaning, different words (illustrative sentences)
q = "Am I allowed to bring my cell to the test?"
cands = ["Students are forbidden to bring mobile phones into the examination hall.",
         "Borrowing entitlement: Two books for ten days.",
         "The fee for re-admission is 25% of the total fee."]
sims = embed(cands) @ embed([q])[0]
print(f"\nQ: {q}")
for c, s in sorted(zip(cands, sims), key=lambda x: -x[1]):
    print(f"  cos = {s:.2f} | {c}   (shared keywords: {sorted(set(tokenize(q)) & set(tokenize(c)))})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded sentence-transformers/all-MiniLM-L6-v2: 384-dim embeddings, max 256 tokens

Q: Am I allowed to bring my cell to the test?
  cos = 0.51 | Students are forbidden to bring mobile phones into the examination hall.   (shared keywords: ['bring'])
  cos = 0.10 | Borrowing entitlement: Two books for ten days.   (shared keywords: [])
  cos = 0.02 | The fee for re-admission is 25% of the total fee.   (shared keywords: [])


In [59]:
def minmax(x):
    """rescale scores to [0, 1] per query so BM25 and cosine can be mixed"""
    x = np.asarray(x, dtype=float)
    span = x.max() - x.min()
    return (x - x.min()) / span if span > 0 else np.zeros_like(x)


class SemanticQA:
    """Week-3 pipeline v1: hybrid retrieval + semantic answer selection + lexical/semantic confidence."""
    NOT_FOUND = KeywordQA.NOT_FOUND

    def __init__(self, chunks, beta=0.5, threshold=0.0, k=5):
        self.chunks = chunks.reset_index(drop=True)
        self.ids = self.chunks.chunk_id.tolist()
        self.beta, self.threshold, self.k = beta, threshold, k
        # lexical side (same as Week 2)
        self.bm25 = BM25Okapi([tokenize(t) for t in self.chunks.text])
        doc_freq = Counter(t for text in self.chunks.text for t in set(tokenize(text)))
        n = len(self.chunks)
        self.idf = {t: np.log((n + 1) / (df + 1)) + 1 for t, df in doc_freq.items()}
        self.unseen_idf = np.log(n + 1) + 1
        # semantic side: one embedding per sentence, prefixed with its section title
        pairs = [(ci, s) for ci, text in enumerate(self.chunks.text) for s in split_sentences(text)]
        self.sent_chunk = np.array([ci for ci, _ in pairs])
        self.sent_text = [s for _, s in pairs]
        sections = self.chunks.section.tolist()
        self.sent_inputs = [f"{sections[ci]}: {s}" for ci, s in pairs]
        self.sent_emb = embed(self.sent_inputs)
        assert set(self.sent_chunk) == set(range(n)), "every chunk needs at least one sentence"

    # ---------- retrieval ----------
    def _scores(self, question):
        sent_sim = self.sent_emb @ embed([question])[0]                 # cosine to every sentence
        semantic = np.full(len(self.ids), -1.0)
        np.maximum.at(semantic, self.sent_chunk, sent_sim)              # best sentence per chunk
        lexical = self.bm25.get_scores(tokenize(question))
        return (1 - self.beta) * minmax(lexical) + self.beta * minmax(semantic), sent_sim

    def search(self, question, k=5):
        score, _ = self._scores(question)
        top = np.argsort(-score, kind="stable")[:k]
        return [(self.ids[i], float(score[i])) for i in top]

    # ---------- answer + confidence ----------
    def select_answer(self, ci, sent_sim):
        """most similar sentence of chunk ci (verbatim); add the next one if it is very short"""
        idx = np.flatnonzero(self.sent_chunk == ci)
        best = int(np.argmax(sent_sim[idx]))
        sents, passage = [self.sent_text[j] for j in idx], self.chunks.text[ci]
        answer = sents[best]
        if len(answer.split()) < 8 and best + 1 < len(sents):           # same rule as the baseline
            start = passage.find(answer)
            answer = passage[start: passage.find(sents[best + 1], start) + len(sents[best + 1])]
        return answer, float(sent_sim[idx[best]])

    def keyword_coverage(self, question, passage):
        """Week-2 confidence: IDF-weighted share of question keywords found in the passage"""
        q_terms = set(tokenize(question))
        weight = {t: self.idf.get(t, self.unseen_idf) for t in q_terms}
        found = q_terms & set(tokenize(passage))
        return sum(weight[t] for t in found) / sum(weight.values())

    def ask(self, question):
        if not tokenize(question):                                       # empty / only stop-words
            return {"answer": self.NOT_FOUND, "source": "", "confidence": 0.0, "supporting_text": "",
                    "extracted": "", "abstained": True, "cited_page": None, "ranked_ids": [],
                    "lexical_coverage": 0.0, "semantic_similarity": 0.0}
        score, sent_sim = self._scores(question)
        order = np.argsort(-score, kind="stable")[:self.k]
        ci = int(order[0])
        top = self.chunks.iloc[ci]
        extracted, semantic = self.select_answer(ci, sent_sim)
        lexical = self.keyword_coverage(question, top.text)
        confidence = round((lexical + max(semantic, 0.0)) / 2, 3)      # same value is used to decide and to report
        abstained = confidence < self.threshold
        return {
            "answer": self.NOT_FOUND if abstained else extracted,
            "source": f"SRB 2026 | {top.part} | {top.section} | PDF page {top.page}",
            "confidence": confidence,
            "supporting_text": top.text,
            # extra fields used for evaluation / inspection
            "extracted": extracted, "abstained": abstained,
            "cited_page": int(top.page), "ranked_ids": [self.ids[i] for i in order],
            "lexical_coverage": round(lexical, 3), "semantic_similarity": round(semantic, 3),
        }

In [60]:
V1_SIZE, V1_BETA, V1_THRESHOLD = 100, 0.6, 0.48          # frozen in Week 3
chunks_v1 = make_chunks(segments, V1_SIZE, overlap=max(20, V1_SIZE // 4))
if os.path.exists(f"{WEEK3_DIR}/v1_chunks.csv"):          # same chunks as the saved Week-3 file?
    _saved = pd.read_csv(f"{WEEK3_DIR}/v1_chunks.csv")
    print("Chunks identical to week3_outputs/v1_chunks.csv:", _saved.text.tolist() == chunks_v1.text.tolist() and _saved.page.tolist() == chunks_v1.page.tolist())
v1 = SemanticQA(chunks_v1, beta=V1_BETA, threshold=V1_THRESHOLD)
res_test_v1 = run_eval(v1, test, chunks_v1, "final-test")

WEEK3_TEST = {"Hit@1": 0.741, "Hit@3": 0.926, "Hit@5": 0.981, "MRR@5": 0.831, "Citation accuracy (page)": 0.907,
              "Answer accuracy (key fact)": 0.593, "False abstention (answerable)": 0.019,
              "Hallucination rate (out-of-scope)": 0.333}
v1_summary = summarize(res_test_v1)
v1_mismatch = {k: (round(v1_summary[k], 3), v) for k, v in WEEK3_TEST.items() if abs(v1_summary[k] - v) > 0.0015}
print("v1 reproduces the Week-3 test results exactly." if not v1_mismatch else f"WARNING: v1 differs from Week 3 (library versions?): {v1_mismatch}")
pd.DataFrame({"baseline (test)": base_summary, "v1 (test)": v1_summary}).round(3)

Chunks identical to week3_outputs/v1_chunks.csv: True
v1 reproduces the Week-3 test results exactly.


,baseline (test),v1 (test)
Hit@1,0.667,0.741
Hit@3,0.889,0.926
Hit@5,0.926,0.981
MRR@5,0.772,0.831
Citation accuracy (page),0.815,0.907
Answer accuracy (key fact),0.370,0.593
Answer accuracy if never refusing,0.500,0.593
Key fact in supporting text,0.796,0.815
False abstention (answerable),0.222,0.019
Hallucination rate (out-of-scope),0.167,0.333


## 3. v2 components

**`Embedder`**: wraps a sentence-embedding model, L2-normalises, caches every text, and adds the retrieval instruction prefix to *questions* for the bge models (as recommended by their authors; documents get no prefix).

**`SentenceReranker`**: a **cross-encoder** reads the question and a candidate sentence *together* and outputs a relevance logit. This is more accurate than comparing two independent vectors, but too slow to run on every sentence in the corpus, so it only re-scores the 10 candidate chunks that hybrid retrieval returns (retrieve, then rerank).

**`RerankQA.ask`**
1. Hybrid score per chunk (as in v1): `(1-beta) * minmax(BM25) + beta * minmax(best-sentence cosine)`; keep the top 10 chunks.
2. Cross-encoder score for every sentence of those 10 chunks; chunk score = best sentence.
3. Final chunk score = `alpha * minmax(hybrid) + (1-alpha) * minmax(cross-encoder)`.
4. Answer = the highest cross-encoder sentence among the top `M` chunks (copied verbatim); the cited chunk is the chunk that sentence belongs to, and it becomes rank 1.
5. Confidence from three signals: keyword coverage, bi-encoder cosine, cross-encoder probability (calibrated in §5).

In [61]:
BGE_PREFIX_NOTE = "bge: instruction prefix on questions only"

class Embedder:
    def __init__(self, name, query_prefix=""):
        self.name, self.query_prefix = name, query_prefix
        self.model = SentenceTransformer(name, device="cpu"); self.model.eval()
        self._doc, self._qry = {}, {}
    def _encode(self, texts):
        return self.model.encode(list(texts), batch_size=64, normalize_embeddings=True,
                                 convert_to_numpy=True, show_progress_bar=False)
    def docs(self, texts):
        texts = list(texts)
        missing = [t for t in dict.fromkeys(texts) if t not in self._doc]
        if missing:
            for t, v in zip(missing, self._encode(missing)):
                self._doc[t] = v
        return np.stack([self._doc[t] for t in texts])
    def query(self, q):
        if q not in self._qry:
            self._qry[q] = self._encode([self.query_prefix + q])[0]
        return self._qry[q]


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))


class SentenceReranker:
    def __init__(self, name):
        self.name = name
        self.model = CrossEncoder(name, device="cpu", max_length=256)
        self.cache = {}
    def _predict(self, pairs):
        ident = torch.nn.Identity()
        try:                                                   # sentence-transformers >= 3
            s = self.model.predict(pairs, batch_size=32, show_progress_bar=False, activation_fn=ident)
        except TypeError:                                      # older versions
            s = self.model.predict(pairs, batch_size=32, show_progress_bar=False, activation_fct=ident)
        return np.asarray(s, dtype=float).reshape(-1)          # raw logits
    def score(self, question, texts):
        missing = [t for t in dict.fromkeys(texts) if (question, t) not in self.cache]
        if missing:
            for t, s in zip(missing, self._predict([(question, t) for t in missing])):
                self.cache[(question, t)] = float(s)
        return np.array([self.cache[(question, t)] for t in texts])

In [62]:
# v2 sentence splitter: the Week-3 splitter ends a sentence after "(i.e." when a quote mark follows.
# Protect common abbreviations (v1 keeps the original splitter so that its Week-3 numbers reproduce exactly).
_ABBR = re.compile(r"\b(i\.e|e\.g|viz|vs|cf|Dr|Mr|Mrs|Ms|Prof|Sr|approx)\.", re.I)
def split_sentences_v2(text):
    protected = _ABBR.sub(lambda m: m.group(0)[:-1] + "\u2024", text)
    return [p.replace("\u2024", ".") for p in split_sentences(protected)]


class RerankQA:
    # Week-4 pipeline v2: hybrid retrieval -> cross-encoder rerank -> verbatim answer -> calibrated confidence
    NOT_FOUND = KeywordQA.NOT_FOUND

    def __init__(self, chunks, embedder, reranker=None, beta=0.6, alpha=0.5, n_rerank=10, scope_m=1,
                 answer_by="ce", threshold=0.0, k=5, conf_fn=None):
        self.chunks = chunks.reset_index(drop=True)
        self.ids = self.chunks.chunk_id.tolist()
        self.embedder, self.reranker = embedder, reranker
        self.beta, self.alpha, self.n_rerank, self.scope_m = beta, alpha, n_rerank, scope_m
        self.answer_by, self.threshold, self.k, self.conf_fn = answer_by, threshold, k, conf_fn
        n = len(self.chunks)
        self.bm25 = BM25Okapi([tokenize(t) for t in self.chunks.text])
        doc_freq = Counter(t for text in self.chunks.text for t in set(tokenize(text)))
        self.idf = {t: np.log((n + 1) / (df + 1)) + 1 for t, df in doc_freq.items()}
        self.unseen_idf = np.log(n + 1) + 1
        pairs = [(ci, s) for ci, text in enumerate(self.chunks.text) for s in split_sentences_v2(text)]
        self.sent_chunk = np.array([ci for ci, _ in pairs])
        self.sent_text = [s for _, s in pairs]
        sections = self.chunks.section.tolist()
        self.sent_inputs = [f"{sections[ci]}: {s}" for ci, s in pairs]
        self.sent_emb = embedder.docs(self.sent_inputs)
        self.chunk_sents = [np.flatnonzero(self.sent_chunk == ci) for ci in range(n)]
        assert all(len(ix) for ix in self.chunk_sents), "every chunk needs at least one sentence"

    def keyword_coverage(self, question, passage):
        q_terms = set(tokenize(question))
        weight = {t: self.idf.get(t, self.unseen_idf) for t in q_terms}
        found = q_terms & set(tokenize(passage))
        return sum(weight[t] for t in found) / sum(weight.values())

    def _hybrid(self, question):
        sent_sim = self.sent_emb @ self.embedder.query(question)
        semantic = np.full(len(self.ids), -1.0)
        np.maximum.at(semantic, self.sent_chunk, sent_sim)
        lexical = self.bm25.get_scores(tokenize(question))
        return (1 - self.beta) * minmax(lexical) + self.beta * minmax(semantic), sent_sim

    def _answer_text(self, ci, j):
        idx = self.chunk_sents[ci]
        pos = int(np.flatnonzero(idx == j)[0])
        sents, passage = [self.sent_text[x] for x in idx], self.chunks.text[ci]
        answer = sents[pos]
        if len(answer.split()) < 8 and pos + 1 < len(sents):             # same short-sentence rule as v1
            start = passage.find(answer)
            answer = passage[start: passage.find(sents[pos + 1], start) + len(sents[pos + 1])]
        return answer

    def _rank(self, question):
        hybrid, sent_sim = self._hybrid(question)
        order_h = np.argsort(-hybrid, kind="stable")
        use_ce = self.reranker is not None
        if not use_ce:                                                    # v1-style: hybrid ranking, bi-encoder sentence
            ci = int(order_h[0])
            idx = self.chunk_sents[ci]
            j = int(idx[int(np.argmax(sent_sim[idx]))])
            return order_h[: self.k].tolist(), ci, j, sent_sim, None
        cand = order_h[: max(self.n_rerank, self.k)]
        sidx = np.concatenate([self.chunk_sents[c] for c in cand])
        ce = self.reranker.score(question, [self.sent_inputs[j] for j in sidx])
        owner = self.sent_chunk[sidx]
        chunk_ce = np.array([ce[owner == c].max() for c in cand])
        fused = self.alpha * minmax(hybrid[cand]) + (1 - self.alpha) * minmax(chunk_ce)
        order = cand[np.argsort(-fused, kind="stable")]
        pool = np.flatnonzero(np.isin(owner, order[: self.scope_m]))
        key = ce[pool] if self.answer_by == "ce" else sent_sim[sidx[pool]]
        best = int(pool[int(np.argmax(key))])
        j, ci = int(sidx[best]), int(owner[best])
        ranked = [ci] + [int(c) for c in order if c != ci]
        return ranked[: self.k], ci, j, sent_sim, float(ce[best])

    def best_sentences(self, question, n=3):
        # best sentence of each of the top-n ranked passages (extra context for broad questions)
        ranked, _, _, sent_sim, _ = self._rank(question)
        res = []
        for ci in ranked[:n]:
            idx = self.chunk_sents[ci]
            sc = (self.reranker.score(question, [self.sent_inputs[j] for j in idx]) if self.reranker is not None
                  else sent_sim[idx])
            c = self.chunks.iloc[ci]
            res.append((int(c.page), c.section, self._answer_text(ci, int(idx[int(np.argmax(sc))]))))
        return res

    def ask(self, question):
        if not tokenize(question):
            return {"answer": self.NOT_FOUND, "source": "", "confidence": 0.0, "supporting_text": "", "extracted": "",
                    "abstained": True, "cited_page": None, "ranked_ids": [], "lexical_coverage": 0.0,
                    "semantic_similarity": 0.0, "ce_prob": 0.0}
        ranked, ci, j, sent_sim, ce_logit = self._rank(question)
        top = self.chunks.iloc[ci]
        extracted = self._answer_text(ci, j)
        lexical = self.keyword_coverage(question, top.text)
        semantic = max(float(sent_sim[j]), 0.0)
        ce_prob = float(sigmoid(ce_logit)) if ce_logit is not None else 0.0
        if self.conf_fn is not None:
            confidence = round(float(self.conf_fn(np.array([lexical, semantic, ce_prob]))), 3)
        else:
            confidence = round((lexical + semantic) / 2, 3)               # v1 rule
        abstained = confidence < self.threshold
        return {"answer": self.NOT_FOUND if abstained else extracted,
                "source": f"SRB 2026 | {top.part} | {top.section} | PDF page {top.page}",
                "confidence": confidence, "supporting_text": top.text,
                "extracted": extracted, "abstained": abstained, "cited_page": int(top.page),
                "ranked_ids": [self.ids[c] for c in ranked],
                "lexical_coverage": round(lexical, 3), "semantic_similarity": round(semantic, 3),
                "ce_prob": round(ce_prob, 3)}

In [63]:
t0 = time.time()
reranker = SentenceReranker(CE_MODEL)
ENCODERS = {"MiniLM-L6": ("sentence-transformers/all-MiniLM-L6-v2", ""),
            "bge-small": ("BAAI/bge-small-en-v1.5", BGE_PREFIX),
            "bge-base": ("BAAI/bge-base-en-v1.5", BGE_PREFIX)}
embedders = {}
for name, (model_name, prefix) in ENCODERS.items():
    embedders[name] = Embedder(model_name, prefix)
    print(f"loaded {name} ({embedders[name].model.get_sentence_embedding_dimension()}-dim)")
chunks_v2 = chunks_v1                                    # chunk size 100 / overlap 25 kept from Week 3
print(f"models loaded in {time.time() - t0:.0f}s; cross-encoder: {CE_MODEL}")

# illustration: why a cross-encoder helps (the Week-3 failure for the proposal example)
demo = RerankQA(chunks_v2, embedders["bge-small"], reranker, beta=0.6, alpha=0.5, scope_m=3)
q = "What is the minimum attendance required to appear for the end-semester exam?"
show(v1.ask(q), q + "   [v1]")
show(demo.ask(q), q + "   [v2 demo, untuned]")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\saniy\AppData\Local\Temp\ipykernel_34224\2336033046.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"loaded {name} ({embedders[name].model.get_sentence_embedding_dimension()}-dim)")


loaded MiniLM-L6 (384-dim)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\saniy\AppData\Local\Temp\ipykernel_34224\2336033046.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"loaded {name} ({embedders[name].model.get_sentence_embedding_dimension()}-dim)")


loaded bge-small (384-dim)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\saniy\AppData\Local\Temp\ipykernel_34224\2336033046.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"loaded {name} ({embedders[name].model.get_sentence_embedding_dimension()}-dim)")


loaded bge-base (768-dim)
models loaded in 34s; cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2
Q: What is the minimum attendance required to appear for the end-semester exam?   [v1]
  answer         : Attendance requirements are summarised hereunder: Attendance% (In Individual Courses) Remarks 80% and above In individual courses Eligible to appear for Trimester/Semester/Annual term End Examinations.
  source         : SRB 2026 | Part I (University) | 3. Attendance Rules | PDF page 11
  confidence     : 0.626
  supporting_text: Students who have maintained 80% or higher attendance in individual courses during a trimester/semester/annual term are eligible to appear for the respective trimester/semester/annual term-end examinations. Attendance is...

Q: What is the minimum attendance required to appear for the end-semester exam?   [v2 demo, untuned]
  answer         : Attendance requirements are summarised hereunder: Attendance% (In Individual Courses) Remarks 80% and above In indivi

## 4. Tuning on dev

Grid over encoder (3) x beta (3) x mode. Modes: **no cross-encoder** (hybrid ranking + bi-encoder sentence, i.e. v1 with a different embedder) or **cross-encoder** with fusion weight `alpha` (0 = pure cross-encoder order) and answer pooling `M` (1 = best sentence of the top chunk, 3 = best sentence of the top 3 chunks). Chunk size is fixed at 100.

Selection rule (dev only, same as Week 3): highest answer accuracy with no refusals, then MRR@5, then Hit@1. Ties go to the **earlier** grid row, and the grid is ordered simplest first (small encoder, no cross-encoder), so we never pay for complexity that does not help. With only 36 answerable dev questions one question is 2.8 points, so differences of a point or two between rows are noise; the report says so.

In [64]:
BETAS, ALPHAS, SCOPES = [0.4, 0.6, 0.8], [0.0, 0.3, 0.5, 0.7], [1, 3]
rows, systems = [], {}
t0 = time.time()
for ename, emb in embedders.items():
    system = RerankQA(chunks_v2, emb, reranker)
    systems[ename] = system
    for beta in BETAS:
        configs = [("no cross-encoder", 1.0, 1)] + [("cross-encoder", a, m) for a in ALPHAS for m in SCOPES]
        for mode, alpha, scope_m in configs:
            use_ce = mode == "cross-encoder"
            system.beta, system.alpha, system.scope_m = beta, alpha, scope_m
            system.reranker, system.answer_by = (reranker if use_ce else None), ("ce" if use_ce else "bi")
            s = summarize(run_eval(system, dev, chunks_v2, "tune"))
            rows.append({"encoder": ename, "mode": mode, "beta": beta, "alpha": alpha, "scope_m": scope_m,
                         "Hit@1": s["Hit@1"], "Hit@5": s["Hit@5"], "MRR@5": s["MRR@5"],
                         "Citation acc": s["Citation accuracy (page)"],
                         "Answer acc (no refusals)": s["Answer accuracy if never refusing"]})
    print(f"  {ename} done ({time.time() - t0:.0f}s)")
grid = pd.DataFrame(rows)
ranked_grid = grid.sort_values(["Answer acc (no refusals)", "MRR@5", "Hit@1"], ascending=False, kind="mergesort")
best = ranked_grid.iloc[0]
BEST = {"encoder": best.encoder, "mode": best["mode"], "beta": float(best.beta), "alpha": float(best.alpha),
        "scope_m": int(best.scope_m)}
print("Chosen on dev:", BEST)

dev_best = (ranked_grid.groupby(["encoder", "mode"], sort=False).head(1)
            .sort_values(["encoder", "mode"]).reset_index(drop=True))
display(dev_best.round(3))

  MiniLM-L6 done (61s)
  bge-small done (81s)
  bge-base done (254s)
Chosen on dev: {'encoder': 'MiniLM-L6', 'mode': 'cross-encoder', 'beta': 0.6, 'alpha': 0.7, 'scope_m': 1}


,encoder,mode,beta,alpha,scope_m,Hit@1,Hit@5,MRR@5,Citation acc,Answer acc (no refusals)
0,MiniLM-L6,cross-encoder,0.6,0.7,1,0.889,0.972,0.931,0.972,0.806
1,MiniLM-L6,no cross-encoder,0.6,1.0,1,0.889,0.972,0.924,0.944,0.778
2,bge-base,cross-encoder,0.8,0.5,1,0.861,1.000,0.919,0.972,0.778
3,bge-base,no cross-encoder,0.6,1.0,1,0.889,0.972,0.931,0.972,0.722
4,bge-small,cross-encoder,0.6,0.7,1,0.917,0.972,0.944,0.972,0.778
5,bge-small,no cross-encoder,0.6,1.0,1,0.944,0.972,0.958,1.000,0.750


In [65]:
# reset the shared systems and build the final v2 (threshold/confidence are set in section 5)
USE_CE = BEST["mode"] == "cross-encoder"
v2 = RerankQA(chunks_v2, embedders[BEST["encoder"]], reranker if USE_CE else None, beta=BEST["beta"],
              alpha=BEST["alpha"], scope_m=BEST["scope_m"], answer_by="ce" if USE_CE else "bi")
dev_res_v2 = run_eval(v2, dev, chunks_v2, "tune")
print("Dev (no refusals):", summarize(dev_res_v2)[["Hit@1", "MRR@5", "Citation accuracy (page)",
                                                   "Answer accuracy if never refusing"]].round(3).to_dict())

Dev (no refusals): {'Hit@1': 0.889, 'MRR@5': 0.931, 'Citation accuracy (page)': 0.972, 'Answer accuracy if never refusing': 0.806}


## 5. Calibrated confidence (extended dev set)

Three signals per question, all computed from the chosen answer sentence: **keyword coverage** (IDF-weighted, Week 2), **bi-encoder cosine** (Week 3) and **cross-encoder probability** (new). Candidates for the final confidence:

* the Week-3 rule: mean(lexical, semantic);
* the mean of all three;
* a **logistic regression** over the three signals (class-balanced, because answerable questions outnumber out-of-scope ones about 2:1).

The best candidate is chosen by AUROC on dev_ext. For the regression, AUROC and the threshold are computed on **5-fold out-of-fold** predictions so the model is not graded on data it was fitted on. Threshold rule as in Week 3: maximise balanced accuracy, take the middle of the best range. `dev_ext` has about 18-20 out-of-scope questions instead of 4.

In [66]:
def collect_features(system, df):
    feats = []
    for q in df.question:
        o = system.ask(q)
        feats.append([o["lexical_coverage"], o["semantic_similarity"], o["ce_prob"]])
    return np.array(feats)

def pick_threshold(score, y):
    ts = np.round(np.arange(0, 1.0001, 0.01), 2)
    ba = np.array([((score[y == 1] >= t).mean() + (score[y == 0] < t).mean()) / 2 for t in ts])
    tied = ts[np.isclose(ba, ba.max())]
    return float(tied[len(tied) // 2]), float(ba.max()), (float(tied[0]), float(tied[-1]))

v2.threshold, v2.conf_fn = 0.0, None
y = dev_ext.answerable.values.astype(int)
X = collect_features(v2, dev_ext)
Xf = X if USE_CE else X[:, :2]

cands = {"mean(lexical, semantic) [v1 rule]": X[:, :2].mean(axis=1)}
if USE_CE:
    cands["mean(lexical, semantic, cross-encoder)"] = X.mean(axis=1)
lr_cv = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000)
cands["logistic regression (5-fold out-of-fold)"] = cross_val_predict(
    lr_cv, Xf, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
dev_auc = {k: roc_auc_score(y, v) for k, v in cands.items()}
CONF_CHOICE = max(dev_auc, key=dev_auc.get)                  # ties -> first (simplest)
THRESHOLD2, dev_ba, (t_lo, t_hi) = pick_threshold(cands[CONF_CHOICE], y)

lr_final = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000).fit(Xf, y)
if CONF_CHOICE.startswith("logistic"):
    conf_fn = lambda f: lr_final.predict_proba(f[: Xf.shape[1]].reshape(1, -1))[0, 1]
elif CONF_CHOICE.startswith("mean(lexical, semantic, cross"):
    conf_fn = lambda f: float(np.mean(f))
else:
    conf_fn = None
v2.conf_fn, v2.threshold = conf_fn, THRESHOLD2

print("Dev_ext AUROC by confidence rule:", {k: round(v, 3) for k, v in dev_auc.items()})
print(f"Chosen: {CONF_CHOICE} | threshold {THRESHOLD2} (best range {t_lo}-{t_hi}), dev balanced accuracy {dev_ba:.3f}")
if CONF_CHOICE.startswith("logistic"):
    print("Logistic-regression weights [lexical, semantic, cross-encoder]:", lr_final.coef_.round(2).tolist())

# fair control: re-tune the *v1* threshold on the same extended dev set
v1_dev_conf = np.array([v1.ask(q)["confidence"] for q in dev_ext.question])
V1_THRESHOLD_RETUNED, _, _ = pick_threshold(v1_dev_conf, y)
print(f"v1 threshold: Week-3 {V1_THRESHOLD} -> re-tuned on dev_ext {V1_THRESHOLD_RETUNED}")

fig, ax = plt.subplots(figsize=(7, 3))
sc = np.array([v2.ask(q)["confidence"] for q in dev_ext.question])
bins = np.arange(0, 1.01, 0.05)
ax.hist(sc[y == 1], bins=bins, alpha=0.7, label="answerable (dev_ext)")
ax.hist(sc[y == 0], bins=bins, alpha=0.7, label="out-of-scope (dev_ext)")
ax.axvline(THRESHOLD2, color="k", ls="--", label=f"threshold {THRESHOLD2}")
ax.set_xlabel("v2 confidence"); ax.set_ylabel("questions"); ax.legend(); ax.set_title("v2 confidence on dev_ext (in-sample)")
plt.tight_layout(); plt.show()

Dev_ext AUROC by confidence rule: {'mean(lexical, semantic) [v1 rule]': 0.987, 'mean(lexical, semantic, cross-encoder)': 0.999, 'logistic regression (5-fold out-of-fold)': 0.994}
Chosen: mean(lexical, semantic, cross-encoder) | threshold 0.46 (best range 0.44-0.47), dev balanced accuracy 0.986
v1 threshold: Week-3 0.48 -> re-tuned on dev_ext 0.54


C:\Users\saniy\AppData\Local\Temp\ipykernel_34224\4248366798.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Final evaluation on the held-out test set

All settings are frozen. The test set is used here for the first time for v2. Four systems are compared: the Week-2 baseline, Week-3 v1 (frozen), **v1 with its threshold re-tuned on dev_ext** (control: how much of the refusal gain is only a better threshold?), and v2.

In [67]:
res_test_v2 = run_eval(v2, test, chunks_v2, "final-test")
v1.threshold = V1_THRESHOLD_RETUNED
res_test_v1b = run_eval(v1, test, chunks_v1, "final-test")
v1.threshold = V1_THRESHOLD

answered = res_test_v2[~res_test_v2.abstained]
assert all(p in s for p, s in zip(answered.pred_answer, answered.supporting_text))
print(f"Groundedness: {len(answered)} answered test questions, 100% copied verbatim from the cited chunk")

# latency with fresh caches (cross-encoder and question embeddings are cached during tuning)
reranker.cache.clear(); embedders[BEST["encoder"]]._qry.clear()
t0 = time.time()
for q_ in test.question:
    v2.ask(q_)
LATENCY_MS = 1000 * (time.time() - t0) / len(test)
print(f"v2 latency: {LATENCY_MS:.0f} ms per question on CPU (fresh caches)\n")

v2_summary, v1b_summary = summarize(res_test_v2), summarize(res_test_v1b)
comparison = pd.DataFrame({"Baseline (W2)": base_summary, "v1 (W3)": v1_summary,
                           "v1 + re-tuned thr.": v1b_summary, "v2 (W4)": v2_summary})
comparison["v2 - v1"] = comparison["v2 (W4)"] - comparison["v1 (W3)"]
comparison.round(3)

Groundedness: 53 answered test questions, 100% copied verbatim from the cited chunk
v2 latency: 619 ms per question on CPU (fresh caches)



,Baseline (W2),v1 (W3),v1 + re-tuned thr.,v2 (W4),v2 - v1
Hit@1,0.667,0.741,0.741,0.741,0.000
Hit@3,0.889,0.926,0.926,0.981,0.056
Hit@5,0.926,0.981,0.981,0.981,0.000
MRR@5,0.772,0.831,0.831,0.849,0.018
Citation accuracy (page),0.815,0.907,0.907,0.907,0.000
Answer accuracy (key fact),0.370,0.593,0.500,0.704,0.111
Answer accuracy if never refusing,0.500,0.593,0.593,0.704,0.111
Key fact in supporting text,0.796,0.815,0.815,0.815,0.000
False abstention (answerable),0.222,0.019,0.130,0.056,0.037
Hallucination rate (out-of-scope),0.167,0.333,0.000,0.333,0.000


In [68]:
def mcnemar(base_ok, new_ok):
    base_ok, new_ok = np.asarray(base_ok, bool), np.asarray(new_ok, bool)
    fixed, broken = int((new_ok & ~base_ok).sum()), int((base_ok & ~new_ok).sum())
    p = binomtest(min(fixed, broken), fixed + broken, 0.5).pvalue if fixed + broken else 1.0
    return {"fixed": fixed, "broken": broken, "p-value": round(float(p), 4)}

def paired(res_old, res_new):
    o = res_old[res_old.answerable].set_index("qid"); n = res_new[res_new.answerable].set_index("qid").loc[o.index]
    return {"Hit@1": mcnemar(o["rank"] == 1, n["rank"] == 1),
            "Citation accuracy": mcnemar(o.citation_ok, n.citation_ok),
            "Answer accuracy (key fact)": mcnemar(o.answer_ok, n.answer_ok)}

MCN = {"v2_vs_v1": paired(res_test_v1, res_test_v2), "v2_vs_baseline": paired(res_test_base, res_test_v2)}
for name, d in MCN.items():
    print(name); display(pd.DataFrame(d).T)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
show_rows = ["Hit@1", "MRR@5", "Citation accuracy (page)", "Answer accuracy (key fact)",
             "False abstention (answerable)", "Hallucination rate (out-of-scope)"]
comparison.loc[show_rows, ["Baseline (W2)", "v1 (W3)", "v2 (W4)"]].plot.barh(ax=axes[0], xlim=(0, 1), title="Test set: baseline vs v1 vs v2")
axes[0].invert_yaxis()
by_topic = pd.DataFrame({n: r[r.answerable].groupby("topic").answer_ok.mean()
                         for n, r in [("baseline", res_test_base), ("v1", res_test_v1), ("v2", res_test_v2)]}).sort_values("v2")
by_topic.plot.barh(ax=axes[1], xlim=(0, 1), title="Answer accuracy on test by topic (2-7 questions each)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/v2_comparison.png", dpi=170, bbox_inches="tight"); plt.show()

v2_vs_v1


,fixed,broken,p-value
Hit@1,1.0,1.0,1.0000
Citation accuracy,0.0,0.0,1.0000
Answer accuracy (key fact),7.0,1.0,0.0703


v2_vs_baseline


,fixed,broken,p-value
Hit@1,7.0,3.0,0.3438
Citation accuracy,6.0,1.0,0.1250
Answer accuracy (key fact),19.0,1.0,0.0000


C:\Users\saniy\AppData\Local\Temp\ipykernel_34224\1852525279.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT_DIR}/v2_comparison.png", dpi=170, bbox_inches="tight"); plt.show()


## 7. Ablation and error analysis

### 7.1 Which change helps? (test set, post-hoc, no refusals)
Each variant switches one thing. Nothing here was used to choose settings.

In [69]:
emb_best, emb_mini = embedders[BEST["encoder"]], embedders["MiniLM-L6"]
variants = {
    "A. v1 (MiniLM hybrid, bi-encoder sentence)": v1,
    "B. + better embedding model only (no cross-encoder)": RerankQA(chunks_v2, emb_best, None, beta=BEST["beta"], answer_by="bi"),
    "C. + cross-encoder only (MiniLM bi-encoder)": RerankQA(chunks_v2, emb_mini, reranker, beta=BEST["beta"], alpha=BEST["alpha"], scope_m=BEST["scope_m"]),
    "D. v2 full": v2,
    "E. v2, answer sentence picked by bi-encoder": RerankQA(chunks_v2, emb_best, reranker, beta=BEST["beta"], alpha=BEST["alpha"], scope_m=BEST["scope_m"], answer_by="bi"),
}
if not USE_CE:
    variants["D. v2 full"] = v2
abl_rows = []
for name, system in variants.items():
    s = summarize(run_eval(system, test, chunks_v2, "final-test"))
    abl_rows.append({"name": name, **{k: float(s[k]) for k in ["Hit@1", "MRR@5", "Citation accuracy (page)",
                                                              "Answer accuracy if never refusing"]}})
display(pd.DataFrame(abl_rows).set_index("name").round(3))

# refusal signals: AUROC answerable vs out-of-scope on test
sig = pd.DataFrame([{**{k: v2.ask(q)[k] for k in ["lexical_coverage", "semantic_similarity", "ce_prob", "confidence"]},
                     "v1 confidence": v1.ask(q)["confidence"], "answerable": a} for q, a in zip(test.question, test.answerable)])
AUROC = {"baseline keyword confidence": roc_auc_score(res_test_base.answerable, res_test_base.confidence),
         "v1 confidence": roc_auc_score(sig.answerable, sig["v1 confidence"]),
         "keyword coverage": roc_auc_score(sig.answerable, sig.lexical_coverage),
         "bi-encoder cosine": roc_auc_score(sig.answerable, sig.semantic_similarity),
         "cross-encoder score": roc_auc_score(sig.answerable, sig.ce_prob),
         "v2 confidence": roc_auc_score(sig.answerable, sig.confidence)}
print("Test AUROC (answerable vs out-of-scope):", {k: round(float(v), 3) for k, v in AUROC.items()})

,Hit@1,MRR@5,Citation accuracy (page),Answer accuracy if never refusing
name,,,,
"A. v1 (MiniLM hybrid, bi-encoder sentence)",0.741,0.831,0.907,0.593
B. + better embedding model only (no cross-encoder),0.741,0.831,0.907,0.593
C. + cross-encoder only (MiniLM bi-encoder),0.741,0.849,0.907,0.704
D. v2 full,0.741,0.849,0.907,0.704
"E. v2, answer sentence picked by bi-encoder",0.741,0.849,0.907,0.611


Test AUROC (answerable vs out-of-scope): {'baseline keyword confidence': 0.863, 'v1 confidence': 0.985, 'keyword coverage': 0.884, 'bi-encoder cosine': 0.969, 'cross-encoder score': 0.929, 'v2 confidence': 0.965}


### 7.2 Error analysis (test set)
Same buckets as Weeks 2-3. The `diagnosis` column is computed from the evaluation; **write your own root-cause explanation for each case in the report** (the handout asks for 5-10 errors with reasons).

In [70]:
def buckets(res):
    e = res[res.answerable].copy()
    e["outcome"] = np.select([e.answer_ok, e.abstained, e["rank"] != 1],
                             ["correct", "refused (low confidence)", "retrieval miss (wrong passage ranked first)"],
                             default="right passage, wrong sentence picked")
    return e

order_b = ["correct", "retrieval miss (wrong passage ranked first)", "right passage, wrong sentence picked", "refused (low confidence)"]
eb, e1, e2 = buckets(res_test_base), buckets(res_test_v1), buckets(res_test_v2)
bucket_df = pd.DataFrame({"baseline": eb.outcome.value_counts(), "v1": e1.outcome.value_counts(),
                          "v2": e2.outcome.value_counts()}).reindex(order_b).fillna(0).astype(int)
display(bucket_df)

def diagnose(r):
    if not r.answerable:
        return f"Answered although the SRB has no answer (confidence {r.confidence:.2f} >= threshold {THRESHOLD2:.2f}); closest passage was cited anyway"
    rank = "not in the top 5" if pd.isna(r["rank"]) else f"ranked {int(r['rank'])}"
    key = "is" if r.key_in_passage else "is NOT"
    return f"Gold passage {rank}; key fact {key} in the cited passage"

err_rows = []
for outcome, n_max in [("retrieval miss (wrong passage ranked first)", 4), ("right passage, wrong sentence picked", 4),
                       ("refused (low confidence)", 2)]:
    for _, r in e2[e2.outcome == outcome].head(n_max).iterrows():
        err_rows.append({"question": r.question, "gold": r.gold_answer, "pred": r.pred_answer[:160],
                         "outcome": outcome, "diagnosis": diagnose(r)})
oos2 = res_test_v2[(~res_test_v2.answerable) & (~res_test_v2.abstained)]
for _, r in oos2.head(2).iterrows():
    err_rows.append({"question": r.question, "gold": "(not in the SRB)", "pred": r.pred_answer[:160],
                     "outcome": "hallucination (out-of-scope answered)", "diagnosis": diagnose(r)})
err_df = pd.DataFrame(err_rows)
for r in err_rows:
    print(f"Q: {r['question']}\n   gold: {r['gold']}\n   v2  : {r['pred']}\n   {r['outcome']} | {r['diagnosis']}\n")
print("v2 on all out-of-scope test questions:")
display(res_test_v2[~res_test_v2.answerable][["question", "confidence", "abstained"]])

,baseline,v1,v2
outcome,,,
correct,20,32,38
retrieval miss (wrong passage ranked first),12,12,7
"right passage, wrong sentence picked",10,9,6
refused (low confidence),12,1,3


Q: What is the punishment for trying to bribe an examiner?
   gold: Exclusion from four additional examinations; the money goes to the Students Welfare Fund.
   v2  : 9.1.14 Any method to bribe the examiner/s by attaching currency notes or letters or making an appeal inside the answer book
   retrieval miss (wrong passage ranked first) | Gold passage ranked 3; key fact is NOT in the cited passage

Q: What happens if revaluation changes my marks by more than 20%?
   gold: A second revaluation is done and the better of the two revaluations becomes final.
   v2  : After following the process of re-evaluation under the Grievance Redressal, effect will be given to change in original marks after re-evaluation, if any, as und
   retrieval miss (wrong passage ranked first) | Gold passage ranked 2; key fact is NOT in the cited passage

Q: On what grounds can I take an academic break?
   gold: Serious medical reasons, serious family issues, or financial constraints.
   v2  : If a student has app

,question,confidence,abstained
54,What is the annual tuition fee for B Tech Data Science?,0.607,False
55,What time does the college canteen close?,0.230,True
56,Is there a gym on campus and what are its timings?,0.264,True
57,What is the average placement package for B Tech students?,0.657,False
58,Is there a college shuttle bus from the railway station?,0.202,True
59,How much is the parking fee for two-wheelers on campus?,0.307,True


## 8. Tests against the requirements

Unit tests check each part of the v2 component; requirement tests check the Week-4 deliverable end to end. Structural checks stop the cell if they fail; behavioural checks (e.g. "gibberish is refused") are reported as PASS/WARN because they depend on the learned threshold.

In [72]:
unit = {}
E = emb_best.docs(["attendance rules", "library fine"])
unit["Embedder: L2-normalised, finite, deterministic"] = (np.allclose(np.linalg.norm(E, axis=1), 1, atol=1e-4) and np.isfinite(E).all()
                                                          and np.allclose(E, emb_best._encode(["attendance rules", "library fine"]), atol=1e-4))
s1 = reranker.score("Is a smart watch allowed in the exam hall?", ["Students are forbidden to bring mobile phones and smart watch into the examination hall.",
                                                                    "Borrowing entitlement: Two books for ten days."])
unit["Cross-encoder: finite and ranks the relevant sentence higher"] = bool(np.isfinite(s1).all() and s1[0] > s1[1])
unit["Index: one embedding per sentence, every chunk has a sentence"] = (len(v2.sent_emb) == len(v2.sent_text)
                                                                        and all(len(ix) for ix in v2.chunk_sents))
unit["Splitter: 'i.e.' does not end a sentence"] = len(split_sentences_v2("If a student has failed under any head (i.e. 'Semester End Examination' and/ or 'aggregate'), he/ she shall be deemed to have failed in that subject.")) == 1
out = v2.ask("How many library books can I borrow?")
unit["Search: ranked ids are distinct chunks, at most k"] = len(set(out["ranked_ids"])) == len(out["ranked_ids"]) <= v2.k
unit["Answer: verbatim from the cited chunk"] = out["extracted"] in out["supporting_text"]
unit["Cited chunk is ranked first"] = out["ranked_ids"][0] == chunks_v2.chunk_id[chunks_v2.text == out["supporting_text"]].iloc[0]
unit["Confidence in [0, 1]"] = 0 <= out["confidence"] <= 1
unit["Same question -> same output"] = v2.ask("Can I eat in the classroom?") == v2.ask("Can I eat in the classroom?")
unit["Empty / stop-word-only question -> refuses"] = v2.ask("")["abstained"] and v2.ask("what is the")["abstained"]
unit["Confidence in [0,1] for all dev_ext questions"] = all(0 <= v2.ask(q)["confidence"] <= 1 for q in dev_ext.question)
for k_, ok in unit.items():
    print("PASS" if ok else "FAIL", "-", k_)
assert all(unit.values())

print()
behaviour = {"Gibberish question -> refuses": v2.ask("zxqv blorf wibble")["abstained"],
             "Out-of-scope 'canteen' question -> refuses": v2.ask("What time does the college canteen close?")["abstained"],
             "In-scope library question -> answers": not v2.ask("How many library books can I borrow and for how long?")["abstained"]}
for k_, ok in behaviour.items():
    print("PASS" if ok else "WARN", "-", k_)

PASS - Embedder: L2-normalised, finite, deterministic
PASS - Cross-encoder: finite and ranks the relevant sentence higher
PASS - Index: one embedding per sentence, every chunk has a sentence
PASS - Splitter: 'i.e.' does not end a sentence
PASS - Search: ranked ids are distinct chunks, at most k
PASS - Answer: verbatim from the cited chunk
PASS - Cited chunk is ranked first
PASS - Confidence in [0, 1]
PASS - Same question -> same output
PASS - Empty / stop-word-only question -> refuses
PASS - Confidence in [0,1] for all dev_ext questions

PASS - Gibberish question -> refuses
PASS - Out-of-scope 'canteen' question -> refuses
PASS - In-scope library question -> answers


In [73]:
test_metrics = summarize(res_test_v2)
tuning_qids = set().union(*[q for stage, q in EVAL_LOG if stage == "tune"])
all_out = [v2.ask(q) for q in test.question]
kb_pages = set(pages.page)
req = {
    "Improved approach implemented (cross-encoder / better embeddings) and compared with baseline AND v1": isinstance(v2, RerankQA),
    "v2 runs end to end on every test question": len(all_out) == len(test),
    "Every output has answer, source, confidence, supporting_text": all({"answer", "source", "confidence", "supporting_text"} <= o.keys() for o in all_out),
    "Every source names part, section and a PDF page in the knowledge base": all(o["abstained"] or (o["cited_page"] in kb_pages and f"PDF page {o['cited_page']}" in o["source"]) for o in all_out),
    "Grounded: every answer is copied verbatim from its cited chunk": all(o["abstained"] or o["answer"] in o["supporting_text"] for o in all_out),
    "Refusals use the fixed not-found message": all(o["answer"] == RerankQA.NOT_FOUND for o in all_out if o["abstained"]),
    "All four proposal metrics computed on test": test_metrics[["Hit@1", "MRR@5", "Citation accuracy (page)", "Answer accuracy (key fact)", "Hallucination rate (out-of-scope)"]].notna().all(),
    "Baseline reproduces Week-2 test numbers exactly": not mismatch,
    "v1 reproduces Week-3 test numbers": not v1_mismatch,
    "No test question used for tuning or calibration": tuning_qids <= set(dev_ext.qid) and tuning_qids.isdisjoint(test.qid),
    "Extended-dev questions are disjoint from test": set(dev_ext.question).isdisjoint(set(test.question)),
}
for k_, ok in req.items():
    print("PASS" if ok else "FAIL", "-", k_)
assert all(req.values())

hib = {"Hit@1": True, "MRR@5": True, "Citation accuracy (page)": True, "Answer accuracy (key fact)": True,
       "False abstention (answerable)": False, "Hallucination rate (out-of-scope)": False}
scorecard = pd.DataFrame({"baseline": [base_summary[m] for m in hib], "v1": [v1_summary[m] for m in hib], "v2": [test_metrics[m] for m in hib],
                          "v2 beats v1?": ["yes" if test_metrics[m] != v1_summary[m] and (test_metrics[m] > v1_summary[m]) == h_ else "no" for m, h_ in hib.items()]},
                         index=list(hib)).round(3)
print("\nResult scorecard on test (reported, not asserted):")
scorecard

PASS - Improved approach implemented (cross-encoder / better embeddings) and compared with baseline AND v1
PASS - v2 runs end to end on every test question
PASS - Every output has answer, source, confidence, supporting_text
PASS - Every source names part, section and a PDF page in the knowledge base
PASS - Grounded: every answer is copied verbatim from its cited chunk
PASS - Refusals use the fixed not-found message
PASS - All four proposal metrics computed on test
PASS - Baseline reproduces Week-2 test numbers exactly
PASS - v1 reproduces Week-3 test numbers
PASS - No test question used for tuning or calibration
PASS - Extended-dev questions are disjoint from test

Result scorecard on test (reported, not asserted):


,baseline,v1,v2,v2 beats v1?
Hit@1,0.667,0.741,0.741,no
MRR@5,0.772,0.831,0.849,yes
Citation accuracy (page),0.815,0.907,0.907,no
Answer accuracy (key fact),0.370,0.593,0.704,yes
False abstention (answerable),0.222,0.019,0.056,no
Hallucination rate (out-of-scope),0.167,0.333,0.333,no


## 9. Ask a question

Change `QUESTION` and run the cell. The answer comes from the SRB chunks only (needs sections 0-5 to have been run).

In [79]:
QUESTION = "What is the minimum attendance required to appear for the end-semester exam?"

out = v2.ask(QUESTION)
print("QUESTION  :", QUESTION)
print("ANSWER    :", out["answer"])
print("SOURCE    :", out["source"])
print(f"CONFIDENCE: {out['confidence']}  (refuses below {v2.threshold})")
print("EVIDENCE  :", out["supporting_text"])
print("\nBest sentence from each of the top-3 passages (more context for broad questions):")
for rank, (page, section, sent) in enumerate(v2.best_sentences(QUESTION, 3), 1):
    print(f"  #{rank}  PDF page {page} | {section}\n       {sent}")

QUESTION  : What is the minimum attendance required to appear for the end-semester exam?
ANSWER    : Attendance requirements are summarised hereunder: Attendance% (In Individual Courses) Remarks 80% and above In individual courses Eligible to appear for Trimester/Semester/Annual term End Examinations.
SOURCE    : SRB 2026 | Part I (University) | 3. Attendance Rules | PDF page 11
CONFIDENCE: 0.75  (refuses below 0.46)
EVIDENCE  : Students who have maintained 80% or higher attendance in individual courses during a trimester/semester/annual term are eligible to appear for the respective trimester/semester/annual term-end examinations. Attendance is considered from the date of commencement of each semester/trimester/annual term. In case of any delay in the admission process of any particular student/ lateral admission case, attendance will be considered from the date of such admission. The decision taken by the University will be final and binding upon everyone. Attendance requirements are